In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [2]:
ubuntu_path = "/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv"
mac_path = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path=mac_path,
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df_raw = graph_data.historical_flows_df.copy()

/Users/vladislav/Documents/vlzm/GFDRR/gbp/loaders/dataloader_raw.py:221: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.stations_capacities_df['capacity'] = 100


In [44]:
historical_flows_df_raw

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,hist_176292,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,0,17,<NA>,<NA>,1,<NA>
1,hist_22891,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
2,hist_24113,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
3,hist_81565,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,4,29,<NA>,<NA>,1,<NA>
4,hist_40992,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1992771,hist_182635,0,1,359,user_trip,arrived,classic_bike,5303.08,6004.06,6004.06,342,359,359,<NA>,1,<NA>
1992772,hist_87216,0,1,359,user_trip,arrived,classic_bike,7688.12,7872.03,7872.03,343,359,359,<NA>,1,<NA>
1992773,hist_912617,0,1,359,user_trip,arrived,classic_bike,5581.01,5623.03,5623.03,338,359,359,<NA>,1,<NA>
1992774,hist_920108,0,1,359,user_trip,arrived,classic_bike,5581.01,5623.03,5623.03,338,359,359,<NA>,1,<NA>


In [7]:
# // graph_data.periods_df
# // graph_data.commodities_categories_df
# // graph_data.facilities_df

# // мне нужно сделать датафрейм в котором будут все возможные комбинации периодов, категорий товаров и объектов.
inventory_backbone = graph_data.periods_df[['period_id']].merge(graph_data.commodities_categories_df, how='cross').merge(graph_data.facilities_df[['facility_id']], how='cross')
inventory_backbone

,period_id,commodity_category,facility_id
0,0,classic_bike,6602.05
1,0,classic_bike,5311.08
2,0,classic_bike,6789.08
3,0,classic_bike,6605.08
4,0,classic_bike,5584.04
...,...,...,...
1627195,359,electric_bike,depot_6
1627196,359,electric_bike,depot_7
1627197,359,electric_bike,depot_8
1627198,359,electric_bike,depot_9


In [ ]:
inventory_backbone = (
    graph_data.periods_df[['period_id']]
    .merge(graph_data.commodities_categories_df, how='cross')
    .merge(graph_data.facilities_df[['facility_id']], how='cross')
)

historical_inventory_df_raw = graph_data.historical_inventory_df.copy()
historical_inventory_df = historical_inventory_df_raw.rename(columns={'quantity': 'quantity_eop'})

# Known starting stock (before period 0), per facility/commodity.
initial_inventory = graph_data.initial_inventory_df.rename(columns={'quantity': 'quantity_initial'})

# Complete panel: every period x commodity x facility, plus the starting stock.
historical_inventory_df = pd.merge(
    inventory_backbone, historical_inventory_df,
    on=['period_id', 'facility_id', 'commodity_category'], how='left',
)
historical_inventory_df = historical_inventory_df.merge(
    initial_inventory, on=['facility_id', 'commodity_category'], how='left',
)
historical_inventory_df = historical_inventory_df.sort_values(
    ['facility_id', 'commodity_category', 'period_id']
)

# A missing record means "stock did not change", not "stock is zero".
eop = historical_inventory_df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].ffill()
# Leading gap (before the first record): fall back to the starting stock,
# then to 0 for pairs that never held anything.
historical_inventory_df['quantity_eop'] = (
    eop.fillna(historical_inventory_df['quantity_initial']).fillna(0)
)

# Start of period = end of the previous period; for period 0 it is the
# starting stock, not the end-of-period-0 value.
sop = historical_inventory_df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].shift(1)
historical_inventory_df['quantity_sop'] = (
    sop.fillna(historical_inventory_df['quantity_initial']).fillna(0)
)

historical_inventory_df = historical_inventory_df.drop(columns='quantity_initial').reset_index(drop=True)
historical_inventory_df



,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
0,0,classic_bike,1234.56,11.0,11.0
1,1,classic_bike,1234.56,11.0,11.0
2,2,classic_bike,1234.56,11.0,11.0
3,3,classic_bike,1234.56,11.0,11.0
4,4,classic_bike,1234.56,11.0,11.0
...,...,...,...,...,...
1627195,355,electric_bike,depot_9,0.0,0.0
1627196,356,electric_bike,depot_9,0.0,0.0
1627197,357,electric_bike,depot_9,0.0,0.0
1627198,358,electric_bike,depot_9,0.0,0.0


In [40]:
def build_historical_inventory_wiled_df(
    periods_df,
    commodities_categories_df,
    facilities_df,
    historical_inventory_df,
    initial_inventory_df,
):
    """Build a complete inventory panel from the inventory inputs.

    Returns one row per (period, commodity_category, facility) with the
    end-of-period and start-of-period stock filled in. A missing source
    record means "stock did not change", not "stock is zero".

    Parameters
    ----------
    periods_df : pandas.DataFrame
        Must contain ``period_id``.
    commodities_categories_df : pandas.DataFrame
        Must contain ``commodity_category``.
    facilities_df : pandas.DataFrame
        Must contain ``facility_id``.
    historical_inventory_df : pandas.DataFrame
        Recorded end-of-period stock: ``period_id``, ``facility_id``,
        ``commodity_category``, ``quantity``.
    initial_inventory_df : pandas.DataFrame
        Starting stock (before period 0): ``facility_id``,
        ``commodity_category``, ``quantity``.

    Returns
    -------
    pandas.DataFrame
        Columns: ``period_id``, ``commodity_category``, ``facility_id``,
        ``quantity_eop``, ``quantity_sop``.
    """
    # Full panel: every period x commodity x facility.
    inventory_backbone = (
        periods_df[['period_id']]
        .merge(commodities_categories_df, how='cross')
        .merge(facilities_df[['facility_id']], how='cross')
    )

    historical = historical_inventory_df.rename(columns={'quantity': 'quantity_eop'})
    # Known starting stock (before period 0), per facility/commodity.
    initial = initial_inventory_df.rename(columns={'quantity': 'quantity_initial'})

    df = pd.merge(
        inventory_backbone, historical,
        on=['period_id', 'facility_id', 'commodity_category'], how='left',
    )
    df = df.merge(
        initial, on=['facility_id', 'commodity_category'], how='left',
    )
    df = df.sort_values(['facility_id', 'commodity_category', 'period_id'])

    # A missing record means "stock did not change", not "stock is zero".
    eop = df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].ffill()
    # Leading gap (before the first record): fall back to the starting stock,
    # then to 0 for pairs that never held anything.
    df['quantity_eop'] = eop.fillna(df['quantity_initial']).fillna(0)

    # Start of period = end of the previous period; for period 0 it is the
    # starting stock, not the end-of-period-0 value.
    sop = df.groupby(['facility_id', 'commodity_category'])['quantity_eop'].shift(1)
    df['quantity_sop'] = sop.fillna(df['quantity_initial']).fillna(0)

    df = df.drop(columns='quantity_initial').reset_index(drop=True)
    return df


historical_inventory_wiled_df = build_historical_inventory_wiled_df(
    periods_df=graph_data.periods_df,
    commodities_categories_df=graph_data.commodities_categories_df,
    facilities_df=graph_data.facilities_df,
    historical_inventory_df=graph_data.historical_inventory_df,
    initial_inventory_df=graph_data.initial_inventory_df,
)
historical_inventory_wiled_df

,period_id,commodity_category,facility_id,quantity_eop,quantity_sop
0,0,classic_bike,1234.56,11.0,11.0
1,1,classic_bike,1234.56,11.0,11.0
2,2,classic_bike,1234.56,11.0,11.0
3,3,classic_bike,1234.56,11.0,11.0
4,4,classic_bike,1234.56,11.0,11.0
...,...,...,...,...,...
1627195,355,electric_bike,depot_9,0.0,0.0
1627196,356,electric_bike,depot_9,0.0,0.0
1627197,357,electric_bike,depot_9,0.0,0.0
1627198,358,electric_bike,depot_9,0.0,0.0


In [ ]:
def execute(self, state: SimulationState, resolved: ResolvedModelData,
            period: PeriodRow, config: EnvironmentConfig) -> PhaseResult:
    """Split this period's demand into departures and stockout losses, bounded by inventory."""

    t = period.period_id
    demand = resolved.historical_demand_df
    demand_now = demand[demand["period_id"] == t].copy()
    demand_scale_factor = getattr(config, "demand_scale_factor", 1.0)
    demand_now.loc[:, 'quantity'] = (demand_now['quantity'] * demand_scale_factor).round().astype("Int64")
    if demand_now.empty:
        return PhaseResult.empty(state)
    inventory_before = int(state.state_inventory_df["quantity"].sum())


    available = state.state_inventory_df.rename(columns={"quantity": "available"})
    out = demand_now.merge(available, on=["facility_id", "commodity_category"], how="left")
    out["available"] = out["available"].fillna(0)
    out["departed"] = out[["quantity", "available"]].min(axis=1).astype("int64")
    out["lost"] = (out["quantity"] - out["departed"]).astype("int64")
    assert (out["departed"] <= out["available"]).all(), "departed exceeds available inventory"
    assert (out["lost"] >= 0).all(), "stockout loss is negative"
    departures = out[["facility_id", "commodity_category", "departed", "lost"]]

    deltas = (departures[departures["departed"] > 0].rename(columns={"departed": "delta"})
    [["facility_id", "commodity_category", "delta"]].copy())
    deltas["delta"] = -deltas["delta"]

    out = state.state_inventory_df.merge(deltas, on=["facility_id", "commodity_category"], how="outer")
    out["quantity"] = out["quantity"].fillna(0) + out["delta"].fillna(0)
    inventory = out[["facility_id", "commodity_category", "quantity"]]

    lost_demand = departures[departures["lost"] > 0]

    new_state = (state.with_inventory(inventory)
                    .with_intermediates(departures=departures))
    events = None
    if not lost_demand.empty:
        lost_demand = lost_demand.rename(
            columns={"facility_id": "source_id", "lost": "quantity"}
        )

        losses = lost_demand
        period_id = t  
        reason = "stockout"


        event_id = 0 if reason == "stockout" else 1
        na = pd.Series([pd.NA] * len(losses), index=losses.index)

        events = _typed_events(pd.DataFrame({
                "flow_id":             losses.get("flow_id", na),
                "move_id":             0,
                "event_id":            event_id,
                "period_id":           period_id,
                "flow_type":           "user_trip",
                "event_type":          "lost",
                "commodity_category":  losses["commodity_category"],
                "source_id":           losses["source_id"],
                "planned_target_id":   losses.get("planned_target_id", na),
                "realized_target_id":  pd.NA,
                "start_period":        losses.get("start_period", na),
                "planned_end_period":  losses.get("planned_end_period", na),
                "realized_end_period": pd.NA,
                "resource_id":         pd.NA,
                "quantity":            losses["quantity"],
                "reason":              reason,
            }))

    departed = inventory_before - int(inventory["quantity"].sum())
    assert departed == int(departures["departed"].sum()), "stockout moves no inventory"
    return PhaseResult(new_state, events)